In [9]:
import alpaca_trade_api as tradeapi
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from datetime import datetime
import time
import math

# --- 1. CREDENCIALES Y CONFIGURACIÓN ---
API_KEY = 'PKKGHFJB2WGZXF4KGZIQOPJIVU'        # <--- PON TUS CLAVES
SECRET_KEY = '4wYNvV9TVcFMyGCVRzigEQCMbtxyWExJW4ocKQas2PTC'  # <--- PON TUS CLAVES
BASE_URL = 'https://paper-api.alpaca.markets'

# Configuración del Dinero
CASH_PER_TRADE = 10000  # <--- DÓLARES QUE QUIERES INVERTIR POR OPERACIÓN
SYMBOL_TO_TRADE = 'MSTR'

# Configuración del Modelo
MODEL_PATH = 'trading_bot_v3.pth'
SCALER_PATH = 'scaler.pkl'
WINDOW_SIZE = 60
INPUT_DIM = 6
HIDDEN_DIM = 32
NUM_LAYERS = 1
THRESHOLD_BUY = 0.55
THRESHOLD_SELL = 0.45

# Conexión a la API
api = tradeapi.REST(API_KEY, SECRET_KEY, BASE_URL, api_version='v2')

# Definición de la clase (necesaria para cargar el modelo)
class TradingBotLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim=1):
        super(TradingBotLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0)
        self.dropout = nn.Dropout(0.5) 
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return self.sigmoid(out)

# --- 2. LÓGICA DE ANÁLISIS ---

def get_market_prediction():
    """Devuelve: Probabilidad, Z-Score y PRECIO ACTUAL"""
    print("🧠 Analizando mercado...")
    
    # Descarga Robusta
    try:
        btc_raw = yf.download('BTC-USD', period="6mo", interval="1h", auto_adjust=True, progress=False)
        mstr_raw = yf.download('MSTR', period="6mo", interval="1h", auto_adjust=True, progress=False)
    except Exception as e:
        print(f"Error descarga: {e}")
        return None

    # Helpers
    def extract_close(df):
        if isinstance(df.columns, pd.MultiIndex):
            try: return df.xs('Close', axis=1, level=0).iloc[:, 0]
            except: 
                try: return df.xs('Close', axis=1, level=1).iloc[:, 0]
                except: return df.iloc[:, 0]
        else:
            return df['Close'] if 'Close' in df.columns else df.iloc[:, 0]

    def normalize_index(series):
        if series.index.tz is None: series.index = series.index.tz_localize('UTC')
        else: series.index = series.index.tz_convert('UTC')
        series.index = series.index.floor('h')
        return series[~series.index.duplicated(keep='last')]

    # Proceso
    btc = normalize_index(extract_close(btc_raw))
    mstr = normalize_index(extract_close(mstr_raw))
    df = pd.concat([btc, mstr], axis=1, join='inner')
    df.columns = ['BTC', 'MSTR']
    df = df.dropna()

    if len(df) < 60: return None

    # Feature Engineering
    df['lret_btc'] = np.log(df['BTC'] / df['BTC'].shift(1))
    df['lret_mstr'] = np.log(df['MSTR'] / df['MSTR'].shift(1))
    df['ratio'] = df['MSTR'] / df['BTC']
    df['ratio_mean'] = df['ratio'].rolling(window=60).mean()
    df['ratio_std'] = df['ratio'].rolling(window=60).std()
    df['z_score'] = (df['ratio'] - df['ratio_mean']) / df['ratio_std']
    
    features = ['lret_btc', 'lret_mstr', 'ratio', 'z_score', 'ratio_mean', 'ratio_std']
    df_final = df[features].dropna()
    
    # Inferencia
    try:
        model = TradingBotLSTM(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS)
        model.load_state_dict(torch.load(MODEL_PATH))
        model.eval()
        scaler = joblib.load(SCALER_PATH)
    except:
        print("Error cargando modelo.")
        return None

    last_seq = df_final.iloc[-WINDOW_SIZE:].values
    last_seq_scaled = scaler.transform(last_seq)
    input_tensor = torch.from_numpy(last_seq_scaled).float().unsqueeze(0)
    
    with torch.no_grad():
        prob = model(input_tensor).item()
    
    # DEVOLVEMOS TAMBIÉN EL PRECIO ACTUAL DE MSTR
    current_price = df.iloc[-1]['MSTR']
    return prob, df.iloc[-1]['z_score'], current_price

# --- 3. LÓGICA DE EJECUCIÓN (CON CÁLCULO DE ACCIONES) ---

def execute_trade():
    
    # 1. Obtener predicción
    result = get_market_prediction()
    if result is None: return
    prob, z_score, current_price = result # Ahora recibimos 3 valores
    
    print(f"\n📊 Probabilidad: {prob:.4f} | Z-Score: {z_score:.4f} | Precio MSTR: ${current_price:.2f}")
    
    # 2. Determinar Señal
    target_signal = "NEUTRAL"
    if prob > THRESHOLD_BUY: target_signal = "LONG"
    elif prob < THRESHOLD_SELL: target_signal = "SHORT"
    
    print(f"🎯 Señal Objetivo: {target_signal}")
    
    # 3. Calcular Cantidad de Acciones (Dynamic Sizing)
    # Redondeamos hacia abajo (floor) para asegurar que no nos pasamos del presupuesto
    # y para evitar decimales en Shorts (que a veces dan problemas).
    qty_to_trade = math.floor(CASH_PER_TRADE / current_price)
    
    if qty_to_trade < 1:
        print(f"⚠️ Alerta: Con ${CASH_PER_TRADE} no te llega para 1 acción de MSTR (${current_price}).")
        return

    print(f"💰 Gestión de Riesgo: ${CASH_PER_TRADE} / ${current_price:.2f} = {qty_to_trade} acciones.")

    # 4. Ver Posición Actual
    current_position = "NEUTRAL"
    qty_held = 0
    
    try:
        pos = api.get_position(SYMBOL_TO_TRADE)
        qty_held = int(pos.qty) # qty puede ser negativo si estamos short
        if qty_held > 0: current_position = "LONG"
        elif qty_held < 0: current_position = "SHORT"
        print(f"💼 Posición Actual: {current_position} ({qty_held} acciones)")
    except:
        print("💼 Posición Actual: NEUTRAL (0 acciones)")
        current_position = "NEUTRAL"

    # 5. EJECUCIÓN
    
    if target_signal == current_position:
        print("✅ Ya estamos posicionados correctamente. Fin.")
    
    else:
        print("🔄 Ejecutando rebalanceo...")
        
        # A. Cerrar lo viejo
        if current_position != "NEUTRAL":
            print(f"    -> Cerrando posición {current_position} actual...")
            api.close_position(SYMBOL_TO_TRADE)
            time.sleep(3) 
            
        # B. Abrir lo nuevo
        if target_signal == "LONG":
            print(f"    -> Comprando {qty_to_trade} acciones (LONG)...")
            api.submit_order(SYMBOL_TO_TRADE, qty=qty_to_trade, side='buy', type='market', time_in_force='day')
            print("    🚀 Orden enviada.")
            
        elif target_signal == "SHORT":
            print(f"    -> Vendiendo {qty_to_trade} acciones (SHORT)...")
            api.submit_order(SYMBOL_TO_TRADE, qty=qty_to_trade, side='sell', type='market', time_in_force='day')
            print("    🐻 Orden enviada.")
            
        elif target_signal == "NEUTRAL":
            print("    -> Quedándonos en CASH.")

if __name__ == "__main__":
    print(f"--- BOT ACTIVADO ({datetime.now().strftime('%H:%M:%S')}) ---")
    execute_trade()
    print("--- FIN ---")

--- BOT ACTIVADO (14:57:34) ---
🧠 Analizando mercado...

📊 Probabilidad: 0.4115 | Z-Score: -0.5487 | Precio MSTR: $133.50
🎯 Señal Objetivo: SHORT
💰 Gestión de Riesgo: $10000 / $133.50 = 74 acciones.
💼 Posición Actual: SHORT (-35 acciones)
✅ Ya estamos posicionados correctamente. Fin.
--- FIN ---
